In [ ]:
# Notebook setup
try:
    import google.colab  # type: ignore
    from google.colab import output, drive
    # Authorize colab to access google drive
    drive.mount('/content/drive')
    stage0_path ='/content/drive/My Drive/CIS 5200 Final Project/Data/appliances_stage0_1.2.parquet'
    
    COLAB = True
    !pip -q install pandas pyarrow fastparquet numpy matplotlib lightgbm
    
except:
    COLAB = False
    from IPython import get_ipython  # type: ignore

    ipython = get_ipython()
    assert ipython is not None
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")
    
    stage0_path = '../data/appliances_stage0_1.2.parquet'
finally:
    import gc, os, math, json
    from pathlib import Path
    import numpy as np
    import pandas as pd
    path = Path(stage0_path)
    assert path.exists(), f"File not found: {path}"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# **LGBM Evaluation**

In [7]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [8]:
# 1. Load Stage 0

df_sup = pd.read_parquet(stage0_path)
print("Loaded df_sup:", df_sup.shape)

# Sanity: check we have y_log and fold
assert 'y_log' in df_sup.columns, "y_log missing"
assert 'fold' in df_sup.columns, "fold missing"

hyperparameters = {"n_estimators":[100, 200, 300],
                   "learning_rate":[0.1, 0.01, 0.001, 0.0001],
                   "max_depth":[10,20,30]
                   }


Loaded df_sup: (82596, 1309)


In [ ]:
# Evaluation function
def evaluate_split(name, y_true_log_hours, y_pred_log_hours):
    # y_true_log = np.log1p(y_true_hours)
    # y_pred_log = np.log1p(y_pred_hours)
    
    # mae_log = mean_absolute_error(y_true_log, y_pred_log)
    # rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    mae_h = mean_absolute_error(y_true_log_hours, y_pred_log_hours)
    rmse_h = np.sqrt(mean_squared_error(y_true_log_hours, y_pred_log_hours))

    print(f"\n{name} performance:")
    print(f"  MAE_log  = {mae_h:.4f}")
    print(f"  RMSE_log = {rmse_h:.4f}")
    # print(f"  MAE_h    = {mae_h:.2f} hours ({mae_h/24:.2f} days)")
    # print(f"  RMSE_h   = {rmse_h:.2f} hours ({rmse_h/24:.2f} days)")
    
    return mae_h, rmse_h

### Dataset A: Word embeddings

In [16]:
TEXT_PREFIX = 'MPN_'      # text embeddings
TEXT_COLS = [c for c in df_sup.columns if c.startswith(TEXT_PREFIX)]

print(f"Num review text embedding dims: {len(TEXT_COLS)}")
print(f"Text embedding cols: {TEXT_COLS}")

train_df = df_sup[df_sup['fold'] == 'train'].copy()
valid_df = df_sup[df_sup['fold'] == 'valid'].copy()
test_df  = df_sup[df_sup['fold'] == 'test'].copy()

# print("Train/Valid/Test sizes:", train_df.shape, valid_df.shape, test_df.shape)

# Targets (log-space)
y_train_log = train_df['y_log'].to_numpy()
y_valid_log = valid_df['y_log'].to_numpy()
y_test_log  = test_df['y_log'].to_numpy()

# Targets (hours)
y_train_hours = train_df['y_hours'].to_numpy()
y_valid_hours = valid_df['y_hours'].to_numpy()
y_test_hours  = test_df['y_hours'].to_numpy()

# Features (text-only)
X_train_metadata = train_df[TEXT_COLS].to_numpy(dtype=np.float32)
X_valid_metadata = valid_df[TEXT_COLS].to_numpy(dtype=np.float32)
X_test_metadata  = test_df[TEXT_COLS].to_numpy(dtype=np.float32)

# Free DataFrames (save RAM)
del train_df, valid_df, test_df
gc.collect()

# Scale text features (fit on train only)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train_metadata)  #only firt the scaler on train only
X_valid_scaled = scaler.transform(X_valid_metadata)
X_test_scaled  = scaler.transform(X_test_metadata)

print("Train/Valid/Test sizes:", X_train_scaled.shape, X_valid_scaled.shape, X_test_scaled.shape)

# We can free the unscaled arrays if memory is tight
del X_train_metadata, X_valid_metadata, X_test_metadata

Num review text embedding dims: 768
Text embedding cols: ['MPN_1', 'MPN_2', 'MPN_3', 'MPN_4', 'MPN_5', 'MPN_6', 'MPN_7', 'MPN_8', 'MPN_9', 'MPN_10', 'MPN_11', 'MPN_12', 'MPN_13', 'MPN_14', 'MPN_15', 'MPN_16', 'MPN_17', 'MPN_18', 'MPN_19', 'MPN_20', 'MPN_21', 'MPN_22', 'MPN_23', 'MPN_24', 'MPN_25', 'MPN_26', 'MPN_27', 'MPN_28', 'MPN_29', 'MPN_30', 'MPN_31', 'MPN_32', 'MPN_33', 'MPN_34', 'MPN_35', 'MPN_36', 'MPN_37', 'MPN_38', 'MPN_39', 'MPN_40', 'MPN_41', 'MPN_42', 'MPN_43', 'MPN_44', 'MPN_45', 'MPN_46', 'MPN_47', 'MPN_48', 'MPN_49', 'MPN_50', 'MPN_51', 'MPN_52', 'MPN_53', 'MPN_54', 'MPN_55', 'MPN_56', 'MPN_57', 'MPN_58', 'MPN_59', 'MPN_60', 'MPN_61', 'MPN_62', 'MPN_63', 'MPN_64', 'MPN_65', 'MPN_66', 'MPN_67', 'MPN_68', 'MPN_69', 'MPN_70', 'MPN_71', 'MPN_72', 'MPN_73', 'MPN_74', 'MPN_75', 'MPN_76', 'MPN_77', 'MPN_78', 'MPN_79', 'MPN_80', 'MPN_81', 'MPN_82', 'MPN_83', 'MPN_84', 'MPN_85', 'MPN_86', 'MPN_87', 'MPN_88', 'MPN_89', 'MPN_90', 'MPN_91', 'MPN_92', 'MPN_93', 'MPN_94', 'MPN_95', '

In [17]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train_scaled, y_train_log)
lgb_valid = lgb.Dataset(X_valid_scaled, y_valid_log, reference=lgb_train)

best_params = None
results = []
for n_est in hyperparameters["n_estimators"]:
    for learning_rate in hyperparameters["learning_rate"]:
        for max_depth in hyperparameters["max_depth"]:
            params = {
                "num_iterations": n_est,
                "metric": {"l2", "l1", "rmse"},
                "max_depth": max_depth,
                "num_leaves": 31,
                "learning_rate": learning_rate,
                "verbose": 0,
            }


                
            print("Starting training...")
            # train
            gbm = lgb.train(
                params, lgb_train, num_boost_round=20, valid_sets=lgb_valid, callbacks=[lgb.early_stopping(stopping_rounds=5)]
            )

            print("Starting predicting...")
            # predict
            y_pred = gbm.predict(X_test_scaled, num_iteration=gbm.best_iteration)
            # eval
            mae_log_h, rmse_log_h = evaluate_split("LGBM: Metadata only", y_test_log, y_pred)
            
            results.append([n_est, learning_rate, max_depth, mae_log_h, rmse_log_h])

a_results_df = pd.DataFrame(results)
print("Least RMSE:")
a_results_df.sort_values(4,inplace=True)
print(a_results_df.head(1))
print("Least MAE:")
a_results_df.sort_values(3,inplace=True)
print(a_results_df.head(1))


Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[13]	valid_0's rmse: 1.75605	valid_0's l2: 3.08369	valid_0's l1: 1.44233
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.2533
  RMSE_log = 1.6080
Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[13]	valid_0's rmse: 1.75605	valid_0's l2: 3.08369	valid_0's l1: 1.44233
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.2533
  RMSE_log = 1.6080
Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[13]	valid_0's rmse: 1.75605	valid_0's l2: 3.08369	valid_0's l1: 1.44233
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.2533
  RMSE_log = 1.6080
Starting training...
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0

In [ ]:
df_sup

### Dataset B: Metadata Only

In [18]:
ITEM_COLS = [c for c in ['main_category','price', 
                         'store', 'categories'] if c in df_sup.columns]

REVIEW_COLS = [c for c in ['rating', 'helpful_vote',
                           'verified_purchase', 'text_len', 'text_words',
                           'average_rating', 'rating_number', 'y_hours_prev'] if c in df_sup.columns]

print(f"Num review metadata dims: {len(REVIEW_COLS)}")
print(f"Metadata cols: {REVIEW_COLS}")

train_df = df_sup[df_sup['fold'] == 'train'].copy()
valid_df = df_sup[df_sup['fold'] == 'valid'].copy()
test_df  = df_sup[df_sup['fold'] == 'test'].copy()

# print("Train/Valid/Test sizes:", train_df.shape, valid_df.shape, test_df.shape)

# Targets (log-space)
y_train_log = train_df['y_log'].to_numpy()
y_valid_log = valid_df['y_log'].to_numpy()
y_test_log  = test_df['y_log'].to_numpy()

# Targets (hours)
y_train_hours = train_df['y_hours'].to_numpy()
y_valid_hours = valid_df['y_hours'].to_numpy()
y_test_hours  = test_df['y_hours'].to_numpy()

# Features (text-only)
X_train_metadata = train_df[REVIEW_COLS].to_numpy(dtype=np.float32)
X_valid_metadata = valid_df[REVIEW_COLS].to_numpy(dtype=np.float32)
X_test_metadata  = test_df[REVIEW_COLS].to_numpy(dtype=np.float32)

# Free DataFrames (save RAM)
del train_df, valid_df, test_df
gc.collect()

# Scale text features (fit on train only)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train_metadata)  #only fit the scaler on train only
X_valid_scaled = scaler.transform(X_valid_metadata)
X_test_scaled  = scaler.transform(X_test_metadata)

print("Train/Valid/Test sizes:", X_train_scaled.shape, X_valid_scaled.shape, X_test_scaled.shape)

# We can free the unscaled arrays if memory is tight
del X_train_metadata, X_valid_metadata, X_test_metadata
gc.collect()

Num review metadata dims: 8
Metadata cols: ['rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number', 'y_hours_prev']
Train/Valid/Test sizes: (73895, 8) (6162, 8) (2539, 8)


0

In [19]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train_scaled, y_train_log)
lgb_valid = lgb.Dataset(X_valid_scaled, y_valid_log, reference=lgb_train)

best_params = None
results = []
for n_est in hyperparameters["n_estimators"]:
    for learning_rate in hyperparameters["learning_rate"]:
        for max_depth in hyperparameters["max_depth"]:
            params = {
                "num_iterations": n_est,
                "metric": {"l2", "l1", "rmse"},
                "max_depth": max_depth,
                "num_leaves": 8,
                "learning_rate": learning_rate,
                "verbose": 0,
            }


                
            print("Starting training...")
            # train
            gbm = lgb.train(
                params, lgb_train, num_boost_round=20, valid_sets=lgb_valid, callbacks=[lgb.early_stopping(stopping_rounds=5)]
            )

            print("Starting predicting...")
            # predict
            y_pred = gbm.predict(X_test_scaled, num_iteration=gbm.best_iteration)
            # eval
            mae_log_h, rmse_log_h = evaluate_split("LGBM: Metadata only", y_test_log, y_pred)
            
            results.append([n_est, learning_rate, max_depth, mae_log_h, rmse_log_h])

b_results_df = pd.DataFrame(results)
print("Least RMSE:")
b_results_df.sort_values(4,inplace=True)
print(b_results_df.head(1))
print("Least MAE:")
b_results_df.sort_values(3,inplace=True)
print(b_results_df.head(1))

Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[14]	valid_0's rmse: 1.61253	valid_0's l2: 2.60027	valid_0's l1: 1.26882
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.1914
  RMSE_log = 1.6051
Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[14]	valid_0's rmse: 1.61253	valid_0's l2: 2.60027	valid_0's l1: 1.26882
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.1914
  RMSE_log = 1.6051
Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[14]	valid_0's rmse: 1.61253	valid_0's l2: 2.60027	valid_0's l1: 1.26882
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.1914
  RMSE_log = 1.6051
Starting training...
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0

### Dataset C: metadata + text embeddings

In [20]:
TEXT_PREFIX = 'MPN_'      # text embeddings
ITEM_COLS = [c for c in ['main_category','price', 
                         'store', 'categories'] if c in df_sup.columns]

REVIEW_COLS = [c for c in ['rating', 'helpful_vote',
                           'verified_purchase', 'text_len', 'text_words',
                           'average_rating', 'rating_number'] if c in df_sup.columns]

META_AND_TEXT_COLS = [c for c in df_sup.columns if c.startswith(TEXT_PREFIX) or c in ['rating', 'helpful_vote',
                                                                                      'verified_purchase', 'text_len', 'text_words',
                                                                                      'average_rating', 'rating_number', 'y_hours_prev']]

print(f"Num review metadata dims: {len(META_AND_TEXT_COLS)}")
print(f"Metadata cols: {META_AND_TEXT_COLS}")

train_df = df_sup[df_sup['fold'] == 'train'].copy()
valid_df = df_sup[df_sup['fold'] == 'valid'].copy()
test_df  = df_sup[df_sup['fold'] == 'test'].copy()

# print("Train/Valid/Test sizes:", train_df.shape, valid_df.shape, test_df.shape)

# Targets (log-space)
y_train_log = train_df['y_log'].to_numpy()
y_valid_log = valid_df['y_log'].to_numpy()
y_test_log  = test_df['y_log'].to_numpy()

# Targets (hours)
y_train_hours = train_df['y_hours'].to_numpy()
y_valid_hours = valid_df['y_hours'].to_numpy()
y_test_hours  = test_df['y_hours'].to_numpy()

# Features (text-only)
X_train_metadata = train_df[META_AND_TEXT_COLS].to_numpy(dtype=np.float32)
X_valid_metadata = valid_df[META_AND_TEXT_COLS].to_numpy(dtype=np.float32)
X_test_metadata  = test_df[META_AND_TEXT_COLS].to_numpy(dtype=np.float32)

# Free DataFrames (save RAM)
del train_df, valid_df, test_df
gc.collect()

# Scale text features (fit on train only)
scaler = StandardScaler(with_mean=True, with_std=True)
X_train_scaled = scaler.fit_transform(X_train_metadata)  #only firt the scaler on train only
X_valid_scaled = scaler.transform(X_valid_metadata)
X_test_scaled  = scaler.transform(X_test_metadata)

print("Train/Valid/Test sizes:", X_train_scaled.shape, X_valid_scaled.shape, X_test_scaled.shape)

# We can free the unscaled arrays if memory is tight
del X_train_metadata, X_valid_metadata, X_test_metadata

Num review metadata dims: 776
Metadata cols: ['rating', 'helpful_vote', 'verified_purchase', 'text_len', 'text_words', 'average_rating', 'rating_number', 'MPN_1', 'MPN_2', 'MPN_3', 'MPN_4', 'MPN_5', 'MPN_6', 'MPN_7', 'MPN_8', 'MPN_9', 'MPN_10', 'MPN_11', 'MPN_12', 'MPN_13', 'MPN_14', 'MPN_15', 'MPN_16', 'MPN_17', 'MPN_18', 'MPN_19', 'MPN_20', 'MPN_21', 'MPN_22', 'MPN_23', 'MPN_24', 'MPN_25', 'MPN_26', 'MPN_27', 'MPN_28', 'MPN_29', 'MPN_30', 'MPN_31', 'MPN_32', 'MPN_33', 'MPN_34', 'MPN_35', 'MPN_36', 'MPN_37', 'MPN_38', 'MPN_39', 'MPN_40', 'MPN_41', 'MPN_42', 'MPN_43', 'MPN_44', 'MPN_45', 'MPN_46', 'MPN_47', 'MPN_48', 'MPN_49', 'MPN_50', 'MPN_51', 'MPN_52', 'MPN_53', 'MPN_54', 'MPN_55', 'MPN_56', 'MPN_57', 'MPN_58', 'MPN_59', 'MPN_60', 'MPN_61', 'MPN_62', 'MPN_63', 'MPN_64', 'MPN_65', 'MPN_66', 'MPN_67', 'MPN_68', 'MPN_69', 'MPN_70', 'MPN_71', 'MPN_72', 'MPN_73', 'MPN_74', 'MPN_75', 'MPN_76', 'MPN_77', 'MPN_78', 'MPN_79', 'MPN_80', 'MPN_81', 'MPN_82', 'MPN_83', 'MPN_84', 'MPN_85', 'MPN_

In [21]:
import lightgbm as lgb

lgb_train = lgb.Dataset(X_train_scaled, y_train_log)
lgb_valid = lgb.Dataset(X_valid_scaled, y_valid_log, reference=lgb_train)

best_params = None
results = []
for n_est in hyperparameters["n_estimators"]:
    for learning_rate in hyperparameters["learning_rate"]:
        for max_depth in hyperparameters["max_depth"]:
            params = {
                "num_iterations": n_est,
                "metric": {"l2", "l1", "rmse"},
                "max_depth": max_depth,
                "num_leaves": 31,
                "learning_rate": learning_rate,
                "verbose": 0,
            }


                
            print("Starting training...")
            # train
            gbm = lgb.train(
                params, lgb_train, num_boost_round=20, valid_sets=lgb_valid, callbacks=[lgb.early_stopping(stopping_rounds=5)]
            )

            print("Starting predicting...")
            # predict
            y_pred = gbm.predict(X_test_scaled, num_iteration=gbm.best_iteration)
            # eval
            mae_log_h, rmse_log_h = evaluate_split("LGBM: Metadata only", y_test_log, y_pred)
            
            results.append([n_est, learning_rate, max_depth, mae_log_h, rmse_log_h])

c_results_df = pd.DataFrame(results)
print("Least RMSE:")
c_results_df.sort_values(4,inplace=True)
print(c_results_df.head(1))
print("Least MAE:")
c_results_df.sort_values(3,inplace=True)
print(c_results_df.head(1))

Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[13]	valid_0's rmse: 1.60703	valid_0's l2: 2.58253	valid_0's l1: 1.26161
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.1905
  RMSE_log = 1.6062
Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[13]	valid_0's rmse: 1.60703	valid_0's l2: 2.58253	valid_0's l1: 1.26161
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.1905
  RMSE_log = 1.6062
Starting training...
Training until validation scores don't improve for 5 rounds
Early stopping, best iteration is:
[13]	valid_0's rmse: 1.60703	valid_0's l2: 2.58253	valid_0's l1: 1.26161
Starting predicting...

LGBM: Metadata only performance:
  MAE_log  = 1.1905
  RMSE_log = 1.6062
Starting training...
Training until validation scores don't improve for 5 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0